# 🐼 Pandas pour auditeurs — Cours

**Objectif** : découvrir et *expérimenter* les fonctions les plus classiques de **pandas**
(la bibliothèque Python de manipulation de données) en partant de ce que vous connaissez
déjà dans Excel.

**Public** : auditeurs bancaires, à l'aise avec Excel (sommes automatiques, `RECHERCHEV` / `VLOOKUP`),
mais **non spécialistes de Python**.

**Comment lire ce notebook**
- Chaque cellule grise est du *code*. Cliquez dedans puis appuyez sur **`Maj` + `Entrée`** pour l'exécuter.
- **Exécutez les cellules dans l'ordre, de haut en bas.** Une cellule peut dépendre des précédentes.
- À chaque notion, un encadré vous rappelle l'**équivalent Excel** :

> 💡 **Équivalent Excel** : à quoi cela correspond dans un tableur.

> ℹ️ **Aucune donnée réelle** n'est utilisée. Les données de transactions sont *générées aléatoirement*
> dans ce notebook ; vous pouvez donc tout exécuter et tout casser sans risque.


## Sommaire

1. Mise en route & vocabulaire (`DataFrame`)
2. Générer un jeu de transactions de démonstration
3. Lire & écrire des fichiers (`read_csv`, `read_excel`)
4. Regarder ses données (`head`, `info`, `describe`)
5. Sélectionner colonnes et lignes (`loc`, `iloc`)
6. Filtrer (l'équivalent des *filtres* Excel)
7. Trier (`sort_values`)
8. Créer / calculer des colonnes
9. Agréger : `sum`, `mean`, `count` (≈ `SOMME`, `MOYENNE`, `NB`)
10. Grouper : `groupby` (≈ `SOMME.SI` / `SUMIF`)
11. Tableaux croisés : `pivot_table` (≈ *tableau croisé dynamique*)
12. Rapprocher deux tables : `merge` (≈ `RECHERCHEV` / `VLOOKUP`)
13. Travailler avec les dates
14. Qualité des données : valeurs manquantes & doublons
15. 🔎 Cas d'audit concrets (mini-revue analytique)
16. Exporter ses résultats
17. Listes Python : création, manipulation, compréhensions
18. Lire un PDF et extraire des données avec PyMuPDF


## 1. Mise en route & vocabulaire

On *importe* d'abord les deux outils dont on a besoin. C'est l'équivalent d'« ouvrir Excel »
avant de commencer.

- `pandas` : pour manipuler des tableaux de données. On lui donne traditionnellement le surnom `pd`.
- `numpy` : pour générer des nombres (on s'en sert seulement pour fabriquer le jeu de démo). Surnom `np`.

> 💡 **Équivalent Excel** : lancer Excel. Ici, on charge les fonctionnalités une seule fois.


In [ ]:
import pandas as pd
import numpy as np

# Afficher toutes les colonnes sans les tronquer
pd.set_option("display.max_columns", None)

print("pandas version :", pd.__version__)
print("Tout est prêt ✅")

### Le mot le plus important : `DataFrame`

Un **`DataFrame`** est tout simplement **un tableau** : des **colonnes** (avec un nom) et des **lignes**
(numérotées à partir de 0). C'est l'équivalent exact d'**une feuille Excel** ou d'un **tableau structuré**.

| Vocabulaire pandas | Équivalent Excel |
|---|---|
| `DataFrame` | une feuille / un tableau |
| une `column` (colonne) | une colonne (A, B, C…) mais nommée |
| une `row` (ligne) | une ligne |
| l'`index` | le numéro de ligne (commence à **0**) |
| une `Series` | une seule colonne isolée |

Petit exemple jouet pour visualiser :


In [ ]:
exemple = pd.DataFrame({
    "compte":   ["A001", "A002", "A003"],
    "montant":  [1500, 320, 9800],
    "devise":   ["EUR", "EUR", "USD"],
})

exemple

Remarquez la colonne tout à gauche (0, 1, 2) : c'est l'**index**, l'équivalent du numéro de ligne.
⚠️ En Python, **on compte à partir de 0**, pas de 1.

## 2. Générer un jeu de transactions de démonstration

On fabrique ~500 transactions bancaires fictives. **Vous n'avez pas besoin de comprendre ce code** :
exécutez-le simplement. Dans la vraie vie, ces données viendraient d'un export de votre core banking
ou d'un fichier Excel (voir section 3).

Quelques « pièges » réalistes ont été **glissés volontairement** dans les données (doublons,
valeurs manquantes, montants ronds, montants juste sous un seuil…) pour les cas d'audit de la section 15.


In [ ]:
np.random.seed(42)   # rend le tirage reproductible : tout le monde obtient les mêmes données
n = 500

dates = pd.to_datetime("2024-01-01") + pd.to_timedelta(np.random.randint(0, 365, n), unit="D")

comptes = ["FR7610011000601234567890185", "FR7630004000031234567890143",
           "FR7612548029981234567890161", "FR7620041010051234567890138"]

contreparties = ["ALPHA SARL", "BETA SA", "GAMMA GMBH", "DELTA LTD",
                 "EPSILON SAS", "ZETA BV", "Particulier", "ETA TRADING"]

df = pd.DataFrame({
    "transaction_id":   range(1, n + 1),
    "date":             dates,
    "account_id":       np.random.choice(comptes, n),
    "counterparty":     np.random.choice(contreparties, n),
    "transaction_type": np.random.choice(["Virement", "Prélèvement", "Carte", "Espèces", "Chèque"],
                                         n, p=[0.40, 0.20, 0.20, 0.10, 0.10]),
    "sens":             np.random.choice(["Débit", "Crédit"], n, p=[0.6, 0.4]),
    "channel":          np.random.choice(["Online", "Agence", "ATM", "Mobile"], n),
    "amount":           np.round(np.random.lognormal(mean=6.5, sigma=1.2, size=n), 2),
    "currency":         np.random.choice(["EUR", "USD", "GBP"], n, p=[0.85, 0.10, 0.05]),
    "country":          np.random.choice(["LU", "FR", "DE", "BE", "US", "GB"], n),
})

# --- pièges volontaires pour les cas d'audit ---
# montants "ronds"
df.loc[np.random.choice(df.index, 15, replace=False), "amount"] = \
    np.random.choice([1000, 5000, 10000, 50000], 15)
# montants juste SOUS le seuil de déclaration de 10 000 (structuring)
df.loc[np.random.choice(df.index, 8, replace=False), "amount"] = \
    np.random.choice([9900.0, 9950.0, 9800.0, 9990.0], 8)
# valeurs manquantes
df.loc[np.random.choice(df.index, 12, replace=False), "counterparty"] = np.nan
df.loc[np.random.choice(df.index, 7,  replace=False), "country"] = np.nan
# doublons (même transaction enregistrée deux fois, avec un id différent)
dups = df.sample(6, random_state=1).copy()
dups["transaction_id"] = range(n + 1, n + 1 + len(dups))
df = pd.concat([df, dups], ignore_index=True)

# on mélange l'ordre des lignes pour faire réaliste
df = df.sample(frac=1, random_state=7).reset_index(drop=True)

print("Jeu de données prêt :", df.shape[0], "lignes,", df.shape[1], "colonnes")

**Signification des colonnes**

| Colonne | Description |
|---|---|
| `transaction_id` | identifiant unique de l'opération |
| `date` | date de l'opération |
| `account_id` | IBAN du compte de la banque |
| `counterparty` | contrepartie (donneur d'ordre / bénéficiaire) |
| `transaction_type` | type d'opération |
| `sens` | Débit ou Crédit |
| `channel` | canal (Online, Agence, ATM, Mobile) |
| `amount` | montant de l'opération |
| `currency` | devise |
| `country` | pays de la contrepartie |


## 3. Lire & écrire des fichiers

Dans la pratique, vos données arrivent dans un **fichier** (CSV ou Excel). Pour la démo, on enregistre
d'abord notre jeu dans deux fichiers, puis on montre comment les **relire**.

> 💡 **Équivalent Excel** : *Fichier → Enregistrer sous* (écrire) et *Fichier → Ouvrir* (lire).


In [ ]:
# Écrire (export) -- index=False pour ne pas écrire la colonne des numéros de ligne
df.to_csv("transactions.csv", index=False)
df.to_excel("transactions.xlsx", index=False)   # nécessite le paquet openpyxl
print("Fichiers transactions.csv et transactions.xlsx créés.")

In [ ]:
# Lire (import) un CSV
df_csv = pd.read_csv("transactions.csv")

# Lire un fichier Excel
df_xlsx = pd.read_excel("transactions.xlsx")

print("Lignes lues depuis le CSV   :", len(df_csv))
print("Lignes lues depuis l'Excel  :", len(df_xlsx))

> ℹ️ Si `read_excel` / `to_excel` renvoie une erreur disant qu'il manque **openpyxl**, exécutez une fois,
> dans une cellule, la commande : `!pip install openpyxl` (le point d'exclamation lance une installation).

Pour vos vrais fichiers, il suffira de remplacer le nom :
`pd.read_excel("C:/Users/moi/Bureau/mon_export.xlsx")`.
Les autres sections continuent avec la variable `df`.


## 4. Regarder ses données

Premier réflexe d'auditeur : **regarder** ce qu'on a sous la main.

> 💡 **Équivalent Excel** : faire défiler les premières lignes, regarder l'en-tête, compter les lignes.


In [ ]:
df.head()        # les 5 premières lignes (head = "tête"). df.head(10) pour 10 lignes.

In [ ]:
df.tail(3)       # les 3 dernières lignes (tail = "queue")

In [ ]:
df.shape         # (nombre de lignes, nombre de colonnes)

In [ ]:
df.columns       # la liste des noms de colonnes

In [ ]:
df.info()        # type de chaque colonne + nombre de valeurs non manquantes

`df.describe()` calcule d'un coup les **statistiques** des colonnes numériques
(nombre, moyenne, écart-type, min, quartiles, max).

> 💡 **Équivalent Excel** : `NB`, `MOYENNE`, `MIN`, `MAX`, `ÉCARTYPE`… le tout en une ligne.


In [ ]:
df.describe()

## 5. Sélectionner des colonnes et des lignes

### Une ou plusieurs colonnes

> 💡 **Équivalent Excel** : sélectionner la colonne « montant », ou les colonnes « montant » et « devise ».


In [ ]:
df["amount"].head()                       # UNE colonne (entre crochets, son nom entre guillemets)

In [ ]:
df[["amount", "currency"]].head()          # PLUSIEURS colonnes : une liste [ ... ] de noms

### Des lignes précises avec `.loc` et `.iloc`

- `.iloc[...]` : sélection par **position** (i comme *integer*, le numéro). On compte à partir de 0.
- `.loc[...]` : sélection par **étiquette** (label) d'index et **nom** de colonne.

> 💡 **Équivalent Excel** : aller à une cellule/plage précise, par exemple `B2:C5`.


In [ ]:
df.iloc[0]              # la toute première ligne (position 0)

In [ ]:
df.iloc[0:5]            # les 5 premières lignes (positions 0 à 4)

In [ ]:
# .loc avec un nom de colonne : les colonnes date + montant des 5 premières lignes
df.loc[0:4, ["date", "amount"]]

## 6. Filtrer les lignes (les *filtres* d'Excel)

C'est sans doute l'opération la plus utile en audit : **ne garder que les lignes qui remplissent
une condition**.

Le principe : on écrit une **condition** entre crochets, et pandas ne garde que les lignes où elle est vraie.

> 💡 **Équivalent Excel** : les *filtres automatiques*, ou la fonction `FILTRE()`.


In [ ]:
# Toutes les transactions de plus de 10 000
df[df["amount"] > 10000].head()

In [ ]:
# Toutes les opérations en espèces ( == veut dire "est égal à")
df[df["transaction_type"] == "Espèces"].head()

### Combiner plusieurs conditions

- `&` signifie **ET** (toutes les conditions vraies)
- `|` signifie **OU** (au moins une vraie)
- ⚠️ Chaque condition doit être entre **parenthèses**.


In [ ]:
# Espèces ET montant supérieur à 5 000
df[(df["transaction_type"] == "Espèces") & (df["amount"] > 5000)].head()

In [ ]:
# Pays = US OU GB
df[df["country"].isin(["US", "GB"])].head()       # isin = "fait partie de cette liste"


In [ ]:
# Montant compris entre 9 000 et 10 000
df[df["amount"].between(9000, 10000)].head()

In [ ]:
# Le nom de contrepartie contient "SA" (recherche de texte)
df[df["counterparty"].str.contains("SA", na=False)].head()

> ℹ️ `na=False` dit à pandas d'ignorer les valeurs manquantes pendant la recherche de texte,
> sinon elles provoqueraient une erreur.

Pour **compter** combien de lignes correspondent à un filtre, on enchaîne avec `.shape[0]` ou `len(...)` :


In [ ]:
nb = len(df[df["amount"] > 10000])
print("Transactions > 10 000 :", nb)

## 7. Trier

> 💡 **Équivalent Excel** : *Données → Trier* (croissant / décroissant).


In [ ]:
# Tri par montant DÉCROISSANT (les plus gros en haut)
df.sort_values("amount", ascending=False).head()

In [ ]:
# Tri sur deux colonnes : d'abord par compte, puis par date
df.sort_values(["account_id", "date"]).head()

## 8. Créer / calculer une nouvelle colonne

On crée une colonne en écrivant `df["nouveau_nom"] = ...`.

> 💡 **Équivalent Excel** : ajouter une colonne avec une formule qui se recopie sur toutes les lignes.


In [ ]:
# Convertir tous les montants en EUR (taux fictifs, pour l'exemple)
taux = {"EUR": 1.0, "USD": 0.92, "GBP": 1.17}

df["taux_eur"]   = df["currency"].map(taux)      # map = associer chaque devise à son taux
df["amount_eur"] = (df["amount"] * df["taux_eur"]).round(2)

df[["amount", "currency", "taux_eur", "amount_eur"]].head()

In [ ]:
# Une colonne basée sur une condition : marquer les "gros" montants
df["gros_montant"] = df["amount_eur"] > 10000      # donne True / False
df[["amount_eur", "gros_montant"]].head()

## 9. Agréger : `sum`, `mean`, `count`, `min`, `max`

On applique un calcul à **toute une colonne**.

> 💡 **Équivalent Excel** : `=SOMME(...)`, `=MOYENNE(...)`, `=NB(...)`, `=MIN(...)`, `=MAX(...)`.


In [ ]:
print("Total des montants (EUR) :", df["amount_eur"].sum().round(2))
print("Montant moyen (EUR)       :", df["amount_eur"].mean().round(2))
print("Montant médian (EUR)      :", df["amount_eur"].median().round(2))
print("Plus gros montant (EUR)   :", df["amount_eur"].max())
print("Nombre de transactions    :", df["amount_eur"].count())

`value_counts()` compte combien de fois chaque valeur apparaît : parfait pour une colonne de catégories.

> 💡 **Équivalent Excel** : `NB.SI` répété pour chaque valeur, ou un tableau croisé en comptage.


In [ ]:
df["transaction_type"].value_counts()

## 10. Grouper : `groupby` ≈ `SOMME.SI` / `SUMIF`

`groupby` = « **pour chaque** catégorie, calcule… ». C'est l'outil clé pour les **synthèses**.

> 💡 **Équivalent Excel** : `SOMME.SI` / `SUMIF`, ou un **tableau croisé dynamique** simple.

Lecture du code ci-dessous : *« pour chaque `transaction_type`, fais la `sum` de `amount_eur` »*.


In [ ]:
df.groupby("transaction_type")["amount_eur"].sum().round(2)

In [ ]:
# On peut trier le résultat du plus gros au plus petit
df.groupby("transaction_type")["amount_eur"].sum().round(2).sort_values(ascending=False)

On peut grouper sur **plusieurs** colonnes, et demander **plusieurs** calculs à la fois avec `.agg(...)` :

In [ ]:
# Pour chaque compte : total, moyenne et nombre d'opérations
df.groupby("account_id")["amount_eur"].agg(["sum", "mean", "count"]).round(2)

In [ ]:
# Grouper sur deux niveaux : compte puis sens (Débit/Crédit)
df.groupby(["account_id", "sens"])["amount_eur"].sum().round(2)

## 11. Tableaux croisés : `pivot_table`

Quand on veut une catégorie **en lignes** et une autre **en colonnes**, c'est exactement le
**tableau croisé dynamique** d'Excel.

> 💡 **Équivalent Excel** : *Insertion → Tableau croisé dynamique*.

Ici : `transaction_type` en lignes, `sens` en colonnes, et la **somme** des montants dans les cases.


In [ ]:
pd.pivot_table(
    df,
    index="transaction_type",   # les lignes
    columns="sens",             # les colonnes
    values="amount_eur",        # la valeur à agréger
    aggfunc="sum",              # le calcul : "sum", "mean", "count"...
    fill_value=0,               # remplacer les cases vides par 0
).round(2)

In [ ]:
# Même chose en COMPTANT le nombre d'opérations plutôt qu'en sommant
pd.pivot_table(df, index="channel", columns="sens",
               values="transaction_id", aggfunc="count", fill_value=0)

## 12. Rapprocher deux tables : `merge` ≈ `RECHERCHEV` / `VLOOKUP`

Très fréquent en audit : on a une table d'opérations, et **une autre table** de référence
(p. ex. les caractéristiques des comptes). On veut **ramener** les infos de la 2ᵉ table dans la 1ʳᵉ,
en s'appuyant sur une **colonne commune** (ici `account_id`).

> 💡 **Équivalent Excel** : `RECHERCHEV` / `VLOOKUP` (ou `RECHERCHEX` / `XLOOKUP`).

Créons d'abord une petite table de référence des comptes :


In [ ]:
comptes_ref = pd.DataFrame({
    "account_id":   comptes,    # même variable que dans la section 2
    "titulaire":    ["Alpha Holding", "Beta Industries", "Gamma Trust", "Delta Family Office"],
    "segment":      ["Corporate", "Corporate", "Private", "Private"],
    "niveau_risque": ["Faible", "Moyen", "Élevé", "Moyen"],
})
comptes_ref

On **fusionne** maintenant `df` avec `comptes_ref` sur la colonne commune `account_id`.

- `on="account_id"` : la colonne de rapprochement (la « clé » du `VLOOKUP`).
- `how="left"` : on garde **toutes** les lignes de `df` (la table de gauche) — comme un `VLOOKUP`
  classique qui part de la table principale.


In [ ]:
df_enrichi = df.merge(comptes_ref, on="account_id", how="left")

df_enrichi[["transaction_id", "account_id", "titulaire", "segment", "niveau_risque", "amount_eur"]].head()

Maintenant qu'on a le `segment` et le `niveau_risque`, on peut faire des synthèses dessus —
c'est là que `merge` + `groupby` deviennent puissants :

In [ ]:
# Montant total par niveau de risque
df_enrichi.groupby("niveau_risque")["amount_eur"].sum().round(2).sort_values(ascending=False)

## 13. Travailler avec les dates

La colonne `date` est déjà reconnue comme une vraie date (type *datetime*). On peut alors extraire
facilement l'année, le mois, le jour de la semaine… via l'accesseur `.dt`.

> 💡 **Équivalent Excel** : `ANNEE()`, `MOIS()`, `JOURSEM()`.


In [ ]:
# Vérifier que la colonne est bien une date
df["date"].dtype

In [ ]:
df["annee"]      = df["date"].dt.year
df["mois"]       = df["date"].dt.month            # 1 à 12
df["jour_sem"]   = df["date"].dt.dayofweek        # 0 = lundi ... 6 = dimanche
df["nom_jour"]   = df["date"].dt.day_name()

df[["date", "annee", "mois", "jour_sem", "nom_jour"]].head()

**Évolution mensuelle** des montants — un classique de la revue analytique :

In [ ]:
evolution = df.groupby(df["date"].dt.to_period("M"))["amount_eur"].sum().round(2)
evolution

In [ ]:
# Un petit graphique en barres (optionnel mais parlant)
evolution.plot(kind="bar", figsize=(10, 3), title="Montant total par mois (EUR)");

## 14. Qualité des données : valeurs manquantes & doublons

Étape incontournable en audit : **fiabiliser** la donnée avant de l'analyser.

### Valeurs manquantes (`NaN` = *Not a Number*, une case vide)

> 💡 **Équivalent Excel** : repérer les cellules vides, `NB.VIDE`.


In [ ]:
# Combien de valeurs manquantes par colonne ?
df.isna().sum()

In [ ]:
# Voir les lignes où la contrepartie est manquante
df[df["counterparty"].isna()].head()

In [ ]:
# Deux façons de traiter :
df_sans_na = df.dropna(subset=["counterparty"])           # supprimer ces lignes
df_rempli  = df.fillna({"counterparty": "INCONNU",
                        "country": "INCONNU"})            # remplacer la case vide

print("Lignes après dropna :", len(df_sans_na))
print("Manquants après fillna :", df_rempli[["counterparty", "country"]].isna().sum().sum())

### Doublons

> 💡 **Équivalent Excel** : *Données → Supprimer les doublons*, ou une mise en forme conditionnelle
> « valeurs en double ».

`duplicated()` marque les lignes en double. On peut chercher les doublons sur **tout** ou sur un
**sous-ensemble de colonnes métier** (même date, même compte, même montant, même contrepartie…).


In [ ]:
# Doublons "métier" : même date, compte, contrepartie, type et montant
cles = ["date", "account_id", "counterparty", "transaction_type", "amount"]

doublons = df[df.duplicated(subset=cles, keep=False)]      # keep=False -> garde TOUTES les occurrences
print("Lignes impliquées dans un doublon :", len(doublons))
doublons.sort_values(cles).head(10)

In [ ]:
# Obtenir une table SANS doublons (on ne garde que la 1re occurrence)
df_dedup = df.drop_duplicates(subset=cles, keep="first")
print("Avant :", len(df), "-> Après :", len(df_dedup))

## 15. 🔎 Cas d'audit concrets

On combine maintenant tout ce qui précède pour quelques **tests analytiques** typiques.
⚠️ Ce sont des *exemples pédagogiques* simplifiés, pas une méthodologie de contrôle complète.


### 15.1 — Les 10 plus gros montants
Repérer les opérations les plus significatives.

In [ ]:
df.sort_values("amount_eur", ascending=False).head(10)[
    ["transaction_id", "date", "account_id", "counterparty", "amount_eur", "transaction_type"]
]

### 15.2 — Montants « ronds »
Des montants exactement ronds (multiples de 1 000) peuvent mériter une attention particulière.

In [ ]:
ronds = df[df["amount"] % 1000 == 0]
print("Opérations à montant rond :", len(ronds))
ronds[["transaction_id", "date", "counterparty", "amount", "transaction_type"]].head(10)

### 15.3 — Montants juste sous un seuil (*structuring*)
Opérations entre 9 000 et 9 999, c.-à-d. **juste en dessous** du seuil de déclaration de 10 000 :
un schéma classique de fractionnement à surveiller.

In [ ]:
sous_seuil = df[(df["amount"] >= 9000) & (df["amount"] < 10000)]
print("Opérations juste sous 10 000 :", len(sous_seuil))
sous_seuil[["transaction_id", "date", "account_id", "counterparty", "amount"]].sort_values("amount", ascending=False)

### 15.4 — Opérations le week-end
Activité un samedi/dimanche : parfois inhabituel selon le contexte.

In [ ]:
weekend = df[df["date"].dt.dayofweek >= 5]      # 5 = samedi, 6 = dimanche
print("Opérations le week-end :", len(weekend))
weekend[["transaction_id", "date", "nom_jour", "counterparty", "amount_eur"]].head(10)

### 15.5 — Espèces importantes
Cumul des opérations en espèces, par compte, au-dessus d'un seuil de matérialité.

In [ ]:
especes = df[(df["transaction_type"] == "Espèces") & (df["amount_eur"] > 3000)]
especes.groupby("account_id")["amount_eur"].agg(["count", "sum"]).round(2).sort_values("sum", ascending=False)

## 16. Exporter ses résultats

Une fois un échantillon ou une synthèse obtenue, on l'exporte vers Excel/CSV pour le partager
ou le documenter dans le dossier de travail.

> 💡 **Équivalent Excel** : *Enregistrer sous*.


In [ ]:
# On exporte la table des virements internationaux significatifs (section 15)
zone_locale = ["LU", "FR", "DE", "BE"]
suspect = df[
    (df["transaction_type"] == "Virement")
    & (df["amount_eur"] > 8000)
    & (~df["country"].isin(zone_locale))
].sort_values("amount_eur", ascending=False)

suspect.to_excel("operations_a_revoir.xlsx", index=False)
print("Fichier 'operations_a_revoir.xlsx' créé avec", len(suspect), "lignes.")

## 17. Listes Python : création, manipulation, compréhensions

Avant de travailler avec des PDF (section 18), il est utile de maîtriser les **listes Python** —
la structure de données de base que l'on retrouve partout : noms de colonnes, codes pays,
résultats d'extraction de texte…

> 💡 **Équivalent Excel** : une plage de valeurs dans une seule colonne, ou une énumération
> de valeurs séparées par des virgules dans une formule (`SOMME.SI.ENS(…;"FR";"LU")`).


### 17.1 — Créer et parcourir une liste


In [ ]:
# Créer une liste (entre crochets, valeurs séparées par des virgules)
pays = ['France', 'Luxembourg', 'Panama', 'Chypre', 'Malte', 'Singapour']
print(pays)
print("Nombre d'éléments :", len(pays))

In [ ]:
# Accès par position (l'index commence à 0)
print(pays[0])    # premier élément
print(pays[-1])   # dernier élément
print(pays[1:4])  # tranche : index 1 inclus jusqu'à 4 exclus (Lux, Pan, Chypre)

In [ ]:
# Vérifier la présence d'une valeur
print('Panama' in pays)      # True
print('Allemagne' in pays)   # False
print(pays.index('Chypre'))  # position de 'Chypre' dans la liste

### 17.2 — Ajouter, supprimer, trier


In [ ]:
pays_travail = pays.copy()  # on travaille sur une copie pour ne pas modifier l'original

pays_travail.append('Malte')               # ajouter UN élément à la fin
pays_travail.extend(['Allemagne', 'Belgique'])  # ajouter PLUSIEURS éléments
print("Après ajouts :", pays_travail)

pays_travail.remove('Malte')   # supprimer la PREMIÈRE occurrence d'une valeur
retiré = pays_travail.pop()    # retirer ET récupérer le DERNIER élément
print("Retiré :", retiré)
print("Après suppressions :", pays_travail)

In [ ]:
# Trier
pays_copies = ['Panama', 'France', 'Luxembourg', 'Chypre']

pays_copies.sort()                    # tri alphabétique EN PLACE (modifie la liste)
print("sort() en place :", pays_copies)

pays_original = ['Panama', 'France', 'Luxembourg', 'Chypre']
pays_tries = sorted(pays_original)    # tri SANS modifier l'original
print("sorted() sans modification :", pays_original)
print("Nouvelle liste triée        :", pays_tries)

### 17.3 — Compréhensions de liste

Les **compréhensions** permettent de créer ou filtrer une liste en une seule ligne élégante.
C'est une des constructions les plus utiles de Python.

```
[expression   for élément in itérable   if condition]
```


In [ ]:
pays = ['France', 'Luxembourg', 'Panama', 'Chypre', 'Malte', 'Singapour']

# Transformer chaque élément
pays_maj = [p.upper() for p in pays]
print("Majuscules :", pays_maj)

# Filtrer selon une condition
pays_longs = [p for p in pays if len(p) > 6]
print("Noms > 6 caractères :", pays_longs)

# Combiner transformation ET filtre
codes = [p[:2].upper() for p in pays if p != 'Singapour']
print("Codes 2 lettres (hors Singapour) :", codes)

In [ ]:
# Cas d'audit : filtrer des montants au-dessus d'un seuil
montants = [1200.0, 9800.0, 450.50, 7500.0, 3300.0, 9950.0, 150.0]

gros = [m for m in montants if m > 5000]
print("Montants > 5 000 :", gros)

# Compter directement (sum sur une liste de booléens)
nb_suspects = sum(1 for m in montants if 9000 <= m < 10000)
print("Montants entre 9 000 et 9 999 :", nb_suspects)

### 17.4 — Listes et pandas

Les listes et pandas travaillent souvent ensemble.

> 💡 Une `Series` pandas *est* une liste enrichie (index, nom, méthodes vectorisées).


In [ ]:
import pandas as pd

# Liste → Series pandas
pays_liste = ['France', 'Luxembourg', 'Panama', 'Chypre']
s = pd.Series(pays_liste, name='pays')
print(s)

# Series → liste Python
retour = s.tolist()
print("\nRetour en liste :", retour)

# Les noms de colonnes d'un DataFrame SONT une liste
df_exemple = pd.DataFrame({'a': [1, 2], 'b': [3, 4], 'c': [5, 6]})
colonnes = df_exemple.columns.tolist()
print("\nColonnes :", colonnes)

In [ ]:
# Filtrage avec .isin() : équivalent d'une compréhension mais sur une colonne entière
import numpy as np
np.random.seed(42)
df_tx = pd.DataFrame({
    'pays': np.random.choice(['France', 'Panama', 'Chypre', 'Allemagne', 'Malte'], 10),
    'montant': np.random.randint(500, 15000, 10),
})

pays_risque = ['Panama', 'Chypre', 'Malte']   # une liste Python...
alertes = df_tx[df_tx['pays'].isin(pays_risque)]  # ...utilisée dans .isin()
print("Opérations vers pays à risque :\n", alertes)

## 18. Lire un PDF et extraire des données avec PyMuPDF

En audit, les documents arrivent souvent au format **PDF** : rapports d'inspection,
relevés de compte, confirmations de virement… PyMuPDF (importé comme `fitz`) permet
d'en extraire le texte, puis d'y rechercher des données structurées (dates, codes pays…).

> ℹ️ **Installation** : si le module n'est pas disponible, exécutez une fois :
> `!pip install pymupdf`

> 💡 **Équivalent Excel** : la fonction *Données → Depuis un fichier texte/PDF* d'Excel 365,
> mais ici en code — ce qui permet de traiter des dizaines de fichiers en boucle.


### 18.1 — Générer un PDF de démonstration

On crée d'abord un faux « relevé de correspondance bancaire » pour avoir un fichier de travail.
Dans la pratique, vous recevrez directement le PDF de votre client.


In [ ]:
import fitz  # PyMuPDF

contenu_pdf = [
    "Relevé de Correspondance Bancaire — Exercice 2024",
    "",
    "Établi par : Service Conformité  |  Date d'édition : 15/04/2024",
    "Période couverte : 01/01/2024 – 31/03/2024",
    "",
    "Flux recensés :",
    "",
    "  1.  Virement reçu le 08/01/2024 — Origine : France — Montant : 12 500 EUR",
    "  2.  Virement émis le 22/01/2024 — Destination : Luxembourg — Montant : 8 900 EUR",
    "  3.  Règlement reçu le 03/02/2024 — Origine : Panama — Montant : 45 000 USD",
    "  4.  Virement émis le 14/02/2024 — Destination : Chypre — Montant : 9 950 EUR",
    "  5.  Virement reçu le 27/02/2024 — Origine : Allemagne — Montant : 3 200 EUR",
    "  6.  Règlement émis le 05/03/2024 — Destination : Malte — Montant : 7 800 EUR",
    "  7.  Virement reçu le 19/03/2024 — Origine : France — Montant : 22 000 EUR",
    "  8.  Virement émis le 28/03/2024 — Destination : Singapour — Montant : 15 000 USD",
    "",
    "Total flux entrants  : 82 700 EUR",
    "Total flux sortants  : 41 650 EUR",
    "",
    "Observations : flux #3 (Panama) et #4 (Chypre) signalés pour revue LCB-FT.",
    "Prochaine revue prévue le 30/06/2024.",
]

doc = fitz.open()          # créer un document vide
page = doc.new_page()      # ajouter une page
texte = "\n".join(contenu_pdf)
page.insert_text((60, 60), texte, fontsize=10.5)
doc.save("releve_correspondance.pdf")
doc.close()
print("PDF créé : releve_correspondance.pdf")

### 18.2 — Lire et extraire le texte


In [ ]:
doc = fitz.open("releve_correspondance.pdf")

print(f"Nombre de pages : {doc.page_count}")
print("─" * 60)

# Extraire le texte de chaque page
texte_complet = ""
for num_page, page in enumerate(doc, start=1):
    texte_page = page.get_text()           # tout le texte de la page
    texte_complet += texte_page
    print(f"── Page {num_page} ──")
    print(texte_page[:300])                # aperçu

doc.close()

### 18.3 — Extraire des dates avec une expression régulière (*regex*)

On utilise le module `re` pour chercher tous les motifs **`JJ/MM/AAAA`** dans le texte.


In [ ]:
import re

# Motif regex : deux chiffres, slash, deux chiffres, slash, quatre chiffres
motif_date = r'\d{2}/\d{2}/\d{4}'

dates_trouvees = re.findall(motif_date, texte_complet)
print("Dates trouvées :", dates_trouvees)

# Convertir en objets pandas datetime pour pouvoir les trier/filtrer
dates_pd = pd.to_datetime(dates_trouvees, format='%d/%m/%Y')
print("\nDates pandas :", dates_pd.tolist())

### 18.4 — Extraire des noms de pays par correspondance de liste

On dispose d'une **liste de référence** de pays. On cherche lesquels apparaissent dans le texte.
C'est une utilisation directe des compréhensions de liste vues en section 17.


In [ ]:
pays_reference = [
    'France', 'Luxembourg', 'Panama', 'Chypre', 'Malte',
    'Singapour', 'Allemagne', 'Belgique', 'Suisse', 'Monaco',
    'Émirats arabes unis', 'Îles Caïmans',
]

# Compréhension : garder uniquement les pays qui apparaissent dans le texte
pays_dans_pdf = [p for p in pays_reference if p in texte_complet]
print("Pays mentionnés dans le PDF :", pays_dans_pdf)

### 18.5 — Construire un DataFrame depuis le texte extrait

On combine regex + listes pour reconstituer un tableau structuré.


In [ ]:
# Regex pour extraire les lignes de flux (numérotées 1. à 8.)
motif_flux = r'(\d+)\.\s+(Virement|Règlement)\s+(reçu|émis)\s+le\s+(\d{2}/\d{2}/\d{4})\s+—\s+(?:Origine|Destination)\s+:\s+([\w ]+?)\s+—\s+Montant\s+:\s+([\d ]+\s+\w+)'

lignes = re.findall(motif_flux, texte_complet)
print("Lignes extraites :")
for l in lignes:
    print(" ", l)

In [ ]:
# Construire le DataFrame
if lignes:
    df_pdf = pd.DataFrame(lignes, columns=[
        'num', 'type_flux', 'sens', 'date', 'pays', 'montant_brut'
    ])
    df_pdf['date'] = pd.to_datetime(df_pdf['date'], format='%d/%m/%Y')
    # Nettoyer le montant : supprimer espaces et séparer le chiffre de la devise
    df_pdf['devise']  = df_pdf['montant_brut'].str.extract(r'([A-Z]{3})$')
    df_pdf['montant'] = df_pdf['montant_brut'].str.replace(r'[^\d]', '', regex=True).astype(float)
    df_pdf = df_pdf.drop(columns='montant_brut')
    df_pdf

In [ ]:
# Identifier les flux vers des pays à risque (liste de la section 17)
pays_risque = ['Panama', 'Chypre', 'Malte', 'Singapour']

if lignes:
    df_pdf['alerte_pays'] = df_pdf['pays'].isin(pays_risque)
    print("Flux signalés :")
    print(df_pdf[df_pdf['alerte_pays']][['date', 'type_flux', 'sens', 'pays', 'montant', 'devise']])

## 🎓 Récapitulatif : du tableur à pandas

| Besoin | Excel | pandas |
|---|---|---|
| Ouvrir un fichier | *Fichier → Ouvrir* | `pd.read_excel(...)` / `pd.read_csv(...)` |
| Regarder les données | faire défiler | `df.head()`, `df.info()`, `df.describe()` |
| Filtrer | filtres automatiques | `df[df["col"] > x]` |
| Trier | *Données → Trier* | `df.sort_values("col")` |
| Colonne calculée | formule recopiée | `df["nouvelle"] = ...` |
| Somme conditionnelle | `SOMME.SI` / `SUMIF` | `df.groupby("cat")["val"].sum()` |
| Tableau croisé | TCD | `pd.pivot_table(...)` |
| Recherche/jointure | `RECHERCHEV` / `VLOOKUP` | `df.merge(autre, on="clé")` |
| Lire/écrire une liste | `[a, b, c]`, `append`, `pop`… | liste Python |
| Compréhension de liste | `[x for x in l if cond]` | liste Python |
| Lire un PDF | `fitz.open(...)` / `.get_text()` | PyMuPDF (`fitz`) |
| Extraire des données (regex) | `re.findall(motif, texte)` | module `re` |
| Supprimer doublons | *Supprimer les doublons* | `df.drop_duplicates()` |
| Cellules vides | repérage manuel | `df.isna()`, `df.fillna()`, `df.dropna()` |
| Enregistrer | *Enregistrer sous* | `df.to_excel(...)` / `df.to_csv(...)` |

---

> Les sections 17 et 18 couvrent les **listes Python** et la **lecture de PDF** (PyMuPDF).

**Pour aller plus loin** : modifiez les seuils des cas d'audit, changez les colonnes des `groupby`,
testez vos propres filtres. La meilleure façon d'apprendre pandas est de **casser puis réparer** 🙂.
